# 04 Outbreak Investigation Workflow — Exercises

Using the Pine and Cypress Nursing Home Legionnaires' disease line list, work through the SitRep production pipeline yourself.

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (avoid CJK labels rendering as boxes) --
# Scan the system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150


## Question 1: Summary metrics

1. Read in `data/synthetic/legionella_outbreak.csv`
2. Build an `infected` column (`clinical_severity != 'not_ill'`)
3. Compute and print: total residents, infected, attack rate, confirmed, probable, hospitalized, ICU, deaths, CFR

In [ ]:
# TODO: read in the CSV and build the infected column
# TODO: compute all the summary metrics
# TODO: print the structured SitRep summary

## Question 2: The person/time/place trio

1. **Person**: compute the infected cases' median age and proportion male
2. **Time**: draw an epidemic curve (matplotlib bar chart), marking the outbreak period and peak day
3. **Place**: use `groupby(["floor", "wing"])` to compute the attack rate per wing, and print the table

In [ ]:
# TODO: Person — median age, proportion male

In [ ]:
# TODO: Time — epidemic curve + outbreak period + peak day

In [ ]:
# TODO: Place — attack-rate-by-wing table

## Question 3: Stratified summary by age group

1. Build an `age_group` column (60-69 / 70-79 / 80-89 / 90+)
2. Use `groupby("age_group")` to compute per age group: count, infected, attack rate, deaths, CFR
3. Which age group has the highest attack rate? Which has the highest CFR? Are they the same?

In [ ]:
# TODO: build age_group
# TODO: groupby + agg
# TODO: compute AR% and CFR%
# TODO: print the table and answer the questions

## Question 4 (challenge): write a generate_sitrep function

Wrap the logic of Questions 1-3 into a function `generate_sitrep(csv_path)` that returns a dict containing:
- `total_residents`, `infected`, `attack_rate`, `deaths`, `cfr`, `hospitalized`, `icu`
- `peak_date` (the peak day)
- `worst_wing` (the name of the wing with the highest attack rate)

Call the function and print the result.

In [ ]:
# TODO: define the generate_sitrep(csv_path) function
# TODO: call it and print the result

## Question 5: Produce a Word report

Using the `python-docx` package, export the SitRep's summary metrics and epidemic curve to a `.docx` document.

1. Create a Word document with the title "Pine and Cypress Nursing Home Legionnaires' SitRep"
2. Add the report time (`datetime.now()`)
3. Add a summary metrics table (total residents, infected, attack rate, deaths, CFR)
4. Embed the epidemic curve in the document (hint: first save the matplotlib figure as a PNG with `BytesIO`, then use `doc.add_picture()`)
5. Save to `output/my_sitrep.docx`

Hint: install with `pip install python-docx`, and import with `from docx import Document`.

In [ ]:
# TODO: from docx import Document
# TODO: from docx.shared import Inches
# TODO: create Document, add_heading, add_paragraph
# TODO: build the summary metrics table (add_table)
# TODO: save the epidemic curve with BytesIO → doc.add_picture()
# TODO: doc.save("output/my_sitrep.docx")

## Question 6: Exposure analysis for a foodborne cluster (norovirus scenario)

A norovirus cluster broke out after a community banquet. The data below is the guest list (`ate_oysters` indicates whether the guest ate the cold oyster platter).

1. Build an exposure table for the two `ate_oysters` groups (whether they ate the cold oyster platter): group size and case count for each
2. Compute the attack rate for each group, and use `risk_ratio()` to compute the risk ratio (RR)
3. Plot an epidemic curve by `symptom_onset_date` (you may use `plot_epi_curve()` or your own `groupby`)
4. Print a summary: total guests, cases, overall attack rate, RR
5. Based on the shape of the epidemic curve (single peak, rapid decline), determine: is this a point source or a continuous common source cluster? Does the RR result support the oyster platter as the suspected exposure source?

In [ ]:
import numpy as np

# Data: guest list from a community banquet (simulated norovirus foodborne cluster)
rng = np.random.default_rng(614)
n = 240
banquet_date = pd.Timestamp("2026-03-14")

guest_id = np.arange(1, n + 1)
table_no = rng.integers(1, 25, n)  # 24 tables
ate_oysters = rng.random(n) < 0.4  # 40% of guests ate the cold oyster platter

# Guests who ate the cold oyster platter have a markedly higher infection probability (suspected exposure)
p_infect = np.where(ate_oysters, 0.65, 0.08)
infected = rng.random(n) < p_infect

# Norovirus has a short incubation period (about 12-48 hours); only cases have an onset date
incubation_hours = rng.normal(30, 8, n).clip(10, 60)
onset_datetime = pd.DatetimeIndex(banquet_date + pd.to_timedelta(incubation_hours, unit="h"))

df6 = pd.DataFrame({
    "guest_id": guest_id,
    "table_no": table_no,
    "ate_oysters": np.where(ate_oysters, "yes", "no"),
    "infected": infected,
    "symptom_onset_date": pd.NaT,
})
df6.loc[infected, "symptom_onset_date"] = onset_datetime[infected].normalize()

# TODO: build an exposure table for the two ate_oysters groups (group size, case count)
# TODO: compute the attack rate for each group, and use risk_ratio() to compute RR
# TODO: plot the epidemic curve by symptom_onset_date (plot_epi_curve or groupby)
# TODO: print a summary (total guests, cases, overall attack rate, RR)
# TODO: answer: is this a point source or continuous common source cluster? Is the oyster platter the suspected exposure source?

## Question 7: Cross-department cluster investigation (COVID-19 scenario)

A company held a department dinner on 3/1, after which a COVID-19 cluster broke out. The data below is the employee roster (`attended_meeting` indicates whether the employee attended the dinner).

1. Use `summarize_by_group()` to summarize case counts and share by `department`
2. Compute the attack rate for each department (infected / total in each department), and sort to find the highest-risk department
3. Plot an epidemic curve by `symptom_onset_date` using `plot_epi_curve()`
4. Compute the risk ratio RR for attendees vs non-attendees
5. Which department has the highest attack rate? Is it related to that department's dinner attendance rate? Does the RR result support the dinner as the transmission hotspot of this cluster?

In [ ]:
import numpy as np

# Data: employee roster after a company department dinner (simulated COVID-19 workplace cluster)
rng = np.random.default_rng(719)
n = 450
departments = ["業務部", "行政部", "IT部", "財務部", "客服部"]
dept_sizes = [150, 80, 70, 60, 90]
department = np.repeat(departments, dept_sizes)

# Attendance rate differs by department (業務部 has the highest attendance rate)
p_meeting = {"業務部": 0.75, "行政部": 0.30, "IT部": 0.15, "財務部": 0.20, "客服部": 0.35}
attended_meeting = np.array([rng.random() < p_meeting[d] for d in department])

# Attendees have a markedly higher infection probability
p_infect = np.where(attended_meeting, 0.55, 0.05)
infected = rng.random(n) < p_infect

# COVID-19 incubation period is about 2-8 days
onset_offset_days = rng.integers(2, 9, n)
meeting_date = pd.Timestamp("2026-03-01")
onset_date = meeting_date + pd.to_timedelta(onset_offset_days, unit="D")

df7 = pd.DataFrame({
    "employee_id": np.arange(1, n + 1),
    "department": department,
    "attended_meeting": np.where(attended_meeting, "yes", "no"),
    "infected": infected,
    "symptom_onset_date": pd.NaT,
})
df7.loc[infected, "symptom_onset_date"] = onset_date[infected]

# TODO: use summarize_by_group() to summarize case counts and share by department (subset to infected cases only)
# TODO: use groupby("department") to compute the attack rate per department, sorted by attack rate
# TODO: plot the epidemic curve by symptom_onset_date using plot_epi_curve()
# TODO: compute the risk ratio RR for the two attended_meeting groups
# TODO: answer: which department has the highest attack rate? Is it related to dinner attendance rate? Does RR support the dinner as the transmission hotspot?

## Question 8 (challenge): Vaccine coverage and serial interval (school measles cluster scenario)

A measles cluster broke out at an elementary school. The data below simulates the within-classroom transmission chain: each case records its infector (`infector_id`) and generation (`generation`), where generation 0 is the community-acquired index case.

1. Compute the attack rate for vaccinated vs unvaccinated students, and use `risk_ratio()` to compute the risk ratio RR
2. Plot an epidemic curve by `symptom_onset_date`, and observe whether cases fall into multiple waves (generations)
3. For each secondary case with an `infector_id`, compute the difference between its onset date and its infector's onset date to estimate the mean and median serial interval
4. Print a SitRep summary (total students, cases, attack rate, RR, serial interval estimate)
5. Answer: is the estimated serial interval close to the literature-reported measles serial interval (about 11-12 days)? Does this school's 92% vaccine coverage reach the measles herd immunity threshold (about 95%)? How does this relate to the cluster's ability to keep spreading?

In [ ]:
import numpy as np

# Data: measles cluster at an elementary school, simulating the within-classroom transmission chain (with generation and infector)
rng = np.random.default_rng(2026)
n_classes = 20
students_per_class = 20
n = n_classes * students_per_class  # 400 students

class_id = np.repeat([f"C{i+1:02d}" for i in range(n_classes)], students_per_class)
student_id = np.arange(1, n + 1)
vaccinated = rng.random(n) < 0.92  # 92% vaccine coverage (below the ~95% measles herd immunity threshold)

df8 = pd.DataFrame({
    "student_id": student_id,
    "class_id": class_id,
    "vaccinated": np.where(vaccinated, "yes", "no"),
})
df8["infected"] = False
df8["symptom_onset_date"] = pd.NaT
df8["infector_id"] = pd.NA
df8["generation"] = pd.NA

start_date = pd.Timestamp("2026-03-02")

# Generation 0: 3 unvaccinated community-acquired index cases
unvacc_idx = df8.index[df8["vaccinated"] == "no"].to_numpy()
primary_idx = rng.choice(unvacc_idx, size=3, replace=False)
for idx in primary_idx:
    df8.loc[idx, "infected"] = True
    df8.loc[idx, "symptom_onset_date"] = start_date + pd.Timedelta(days=int(rng.integers(0, 3)))
    df8.loc[idx, "generation"] = 0

# Simulate the within-classroom transmission chain by generation: each case may infect
# uninfected classmates; vaccinated classmates can still be infected, but at a much lower
# probability due to vaccine efficacy (97%)
vaccine_efficacy = 0.97
p_transmit_unvacc = 0.55  # transmission probability per classmate-pair contact (unvaccinated)
max_generations = 5

current_gen = 0
while current_gen < max_generations:
    infectors = df8[(df8["generation"] == current_gen) & df8["infected"]]
    if infectors.empty:
        break
    for _, case in infectors.iterrows():
        classmates = df8[
            (df8["class_id"] == case["class_id"])
            & (~df8["infected"])
            & (df8.index != case.name)
        ]
        for cm_idx, cm in classmates.iterrows():
            transmit_prob = (
                p_transmit_unvacc * (1 - vaccine_efficacy)
                if cm["vaccinated"] == "yes"
                else p_transmit_unvacc
            )
            if rng.random() < transmit_prob:
                generation_interval = max(7, rng.normal(12, 2))  # serial interval in days
                onset = case["symptom_onset_date"] + pd.Timedelta(days=generation_interval)
                df8.loc[cm_idx, "infected"] = True
                df8.loc[cm_idx, "symptom_onset_date"] = onset
                df8.loc[cm_idx, "infector_id"] = case["student_id"]
                df8.loc[cm_idx, "generation"] = current_gen + 1
    current_gen += 1

# TODO: compute the attack rate for unvaccinated vs vaccinated students, and use risk_ratio() to compute RR
# TODO: plot the epidemic curve by symptom_onset_date (plot_epi_curve), and observe whether there are multiple generational waves
# TODO: for each case with an infector_id, compute the difference between its onset date and its infector's onset date -> estimate the mean and median serial interval
# TODO: print a SitRep summary (total students, cases, attack rate, RR, serial interval estimate)
# TODO: answer: is the serial interval close to the literature's 11-12 days? Does the 92% vaccine coverage reach the herd immunity threshold?